# CP Lab 3 - Multiply Two Matrices Using CUDA


Siddharth Sudhakar - 25901335 - Group 2

In [ ]:
from numba import cuda
import numpy as np

In [ ]:
import time

## CUDA Kernel

In [ ]:
@cuda.jit
def matmul(A, B, C):
    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        tmp = 0.
        for k in range(A.shape[1]):
            tmp += A[row, k] * B[k, col]

        C[row, col] = tmp

## Host Code

In [ ]:
N = 256
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)

C = np.zeros((N, N), dtype=np.float32)

In [ ]:
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

In [ ]:
threadsperblock = (16, 16)
blockspergrid_x = (N + 15) // 16
blockspergrid_y = (N + 15) // 16

In [ ]:
matmul[(blockspergrid_x, blockspergrid_y), threadsperblock](d_A, d_B, d_C)

C = d_C.copy_to_host()

print("Result shape:", C.shape)

Result shape: (256, 256)


## Convert rgb to grayscale based by CPU and GPU and find difference in time

In [ ]:
H, W = 4096, 4096
rgb_host = np.random.rand(H, W, 3).astype(np.float32)
#gray_gpu_host = np.zeros((H, W), dtype=np.float32)

In [ ]:
@cuda.jit
def rgb_to_grayscale_gpu(rgb_img, gray_img):
    x, y = cuda.grid(2)
    if x < rgb_img.shape[0] and y < rgb_img.shape[1]:
        r = rgb_img[x, y, 0]
        g = rgb_img[x, y, 1]
        b = rgb_img[x, y, 2]
        gray_img[x, y] = 0.299*r + 0.587*g + 0.114*b

In [ ]:
gray_cpu = np.zeros((H, W), dtype=np.float32)

start_cpu = time.time()

for i in range(H):
    for j in range(W):
        r = rgb_host[i, j, 0]
        g = rgb_host[i, j, 1]
        b = rgb_host[i, j, 2]
        gray_cpu[i, j] = 0.299*r + 0.587*g + 0.114*b

end_cpu = time.time()

cpu_time = end_cpu - start_cpu
print(f"CPU Time: {cpu_time:.4f} seconds")

CPU Time: 24.9007 seconds


In [ ]:
threads_per_block = (16, 16)

blocks_per_grid_x = (H + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid_y = (W + threads_per_block[1] - 1) // threads_per_block[1]

blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

In [ ]:
start_gpu = time.time()

rgb_device = cuda.to_device(rgb_host)
gray_device = cuda.device_array((H, W), dtype=np.float32)

rgb_to_grayscale_gpu[blocks_per_grid, threads_per_block](
    rgb_device, gray_device
)

cuda.synchronize()
gray_gpu = gray_device.copy_to_host()

end_gpu = time.time()

gpu_time = end_gpu - start_gpu
print(f"GPU Time (with transfer): {gpu_time:.4f} seconds")

GPU Time (with transfer): 0.1857 seconds


In [ ]:
time_diff = cpu_time - gpu_time
print(f"Time difference (CPU - GPU): {time_diff:.4f} seconds")

Time difference (CPU - GPU): 24.7150 seconds


In [ ]:
print("First 5x5 CPU result:\n", gray_cpu[:5, :5])
print("\nFirst 5x5 GPU result:\n", gray_gpu[:5, :5])

First 5x5 CPU result:
 [[0.572611   0.77202964 0.6831136  0.6188807  0.26667827]
 [0.7966946  0.41673356 0.6179516  0.5243302  0.4816301 ]
 [0.3859812  0.588627   0.65255994 0.5216696  0.0753056 ]
 [0.46859723 0.8415112  0.18982156 0.5598642  0.10702637]
 [0.55054724 0.65230227 0.6802825  0.79822403 0.17966244]]

First 5x5 GPU result:
 [[0.572611   0.77202964 0.6831135  0.6188807  0.2666783 ]
 [0.79669464 0.41673353 0.6179516  0.5243302  0.48163006]
 [0.3859812  0.588627   0.65255994 0.5216696  0.07530559]
 [0.4685972  0.8415112  0.18982156 0.5598642  0.10702636]
 [0.55054724 0.6523022  0.6802825  0.79822403 0.17966244]]
